# CAS Exam 5: Case Outstanding Techniques

This notebook demonstrates both case outstanding techniques from your formula sheet, while leveraging `chainladder` functionality as much as possible.

Data used:
- `chainladder/utils/data/friedland_us_industry_auto_case.csv`

Methods shown:
1. Case Outstanding Technique #1 (develop case outstanding + paid-on-case style projection).
2. Case Outstanding Technique #2 (case outstanding projection using paid/reported CDF relationship).


## Formula Sheet Reference

### Technique #1: Case Outstanding Development

**Ratios selected (vol-wtd averages over observation window):**

$$r_{case}(d) = \frac{\text{Case Outstanding}_{d+1}}{\text{Case Outstanding}_d} \quad \text{(Remaining-in-case ratio)}$$

$$r_{paid}(d) = \frac{\text{Incremental Paid}_{d+1}}{\text{Case Outstanding}_d} \quad \text{(Paid-on-case ratio)}$$

**Projection (starting from latest observed age for each AY):**

For each future development age step:
1. Projected incremental paid = Current Case Outstanding × r_paid(d)  
2. Projected next case OS = Current Case Outstanding × r_case(d)
3. Repeat from Step 1 using updated case outstanding
4. Sum all projected incremental paids + add any remaining final case balance

**Ultimate Paid = Latest Cumulative Paid + Σ projected future incremental paids**

### Technique #2: Case Outstanding via CDF Combination

$$\text{Case OS Dev Factor} = 1 + \frac{(\text{Reported CDF} - 1) \times \text{Paid CDF}}{\text{Paid CDF} - \text{Reported CDF}}$$

$$\text{Projected Unpaid} = \text{Latest Case Outstanding} \times \text{Case OS Dev Factor}$$

$$\text{Ultimate Paid} = \text{Latest Cumulative Paid} + \text{Projected Unpaid}$$

**Derivation note:** This formula collapses when Paid CDF = Reported CDF (denominator = 0) — which occurs only if paid and reported develop identically (unusual; investigate as a data issue if it occurs).

### Assumption Comparison

| Assumption | Technique #1 | Technique #2 |
|---|---|---|
| Case adequacy is stable | Critical — r_case(d) reflects current adequacy level | Important — CDF_rep depends on stable adequacy |
| Settlement rate is stable | Critical — r_paid(d) reflects current settlement speed | Important — CDF_paid depends on stable settlement |
| Data requirements | Case OS triangle + incremental paid triangle | Paid CDF + reported CDF (from chainladder fit) |
| Sensitivity to individual case balances | High — latest-age case OS drives projection | Lower — CDFs aggregate across origins |

### When to Use Case Outstanding Methods

**Use when:**
- Case outstanding data is available and credible by accident year
- Reported (incurred) development history is not reliable or not available
- You want to explicitly link projected paid claims to current case reserves
- Claim counts are not available (rules out freq-sev and disposal rate methods)

**Do NOT use when:**
- Case adequacy is known to be changing (run Berquist-Sherman adjustment first)
- The case OS triangle is too sparse for credible ratio selection
- Settlement rate changes are suspected but unquantified (diagnose first)

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import chainladder as cl
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import (
    run_case_outstanding_chainladder,
    run_case_outstanding_friedland,
)

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto_case.csv'
raw = pd.read_csv(DATA_PATH).sort_values(['Accident Year', 'Calendar Year']).copy()

raw['Paid Cumulative'] = raw.groupby('Accident Year')['Incremental Paid Claims'].cumsum()
raw['Reported Cumulative'] = raw['Paid Cumulative'] + raw['Case Outstanding']

case_os_triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Case Outstanding'],
    cumulative=False,
)
paid_incremental_triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Incremental Paid Claims'],
    cumulative=False,
)
paid_cumulative_triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Cumulative'],
    cumulative=True,
)
reported_cumulative_triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Reported Cumulative'],
    cumulative=True,
)

{'rows': len(raw), 'case_triangle_shape': case_os_triangle.shape, 'paid_cum_shape': paid_cumulative_triangle.shape}


In [ ]:
latest_case = case_os_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_paid = paid_cumulative_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_reported = reported_cumulative_triangle.latest_diagonal.to_frame().iloc[:, 0]

case_snapshot = pd.DataFrame(
    {
        'LatestCaseOutstanding': latest_case.values,
        'LatestPaidCumulative': latest_paid.values,
        'LatestReportedCumulative': latest_reported.values,
    },
    index=latest_case.index.year,
)
case_snapshot.index.name = 'AccidentYear'
case_snapshot


## Pre-Projection Diagnostic: Is Case Outstanding Stable Enough to Develop?

The case outstanding methods assume case adequacy and settlement rates are stable across origin years. Before projecting, check whether the paid/incurred ratio is trending across origin years using `paid_vs_incurred_comparison`.

In [ ]:
from reservingengine.reserving import paid_vs_incurred_comparison

pi = paid_vs_incurred_comparison(paid_cumulative_triangle, reported_cumulative_triangle)
pi_ratio = pi["paid_to_incurred_latest"]["paid_to_incurred"]
orig_nums = pd.Series(
    [int(str(o)[:4]) if str(o)[:4].isdigit() else float("nan") for o in pi_ratio.index],
    index=pi_ratio.index, dtype=float
)
slope = (
    float(np.polyfit(orig_nums.dropna().values, pi_ratio.dropna().values.astype(float), 1)[0])
    if pi_ratio.dropna().shape[0] >= 2 else float('nan')
)
ca = pd.DataFrame({"paid_to_incurred": pi_ratio})
ca["trend_slope"] = slope

print(f"Case adequacy trend slope across origin years: {ca['trend_slope'].iloc[0]:.6f}")
print()
print("Paid/incurred by accident year (latest diagonal):")
ca.style.format({'paid_to_incurred': '{:.4f}', 'trend_slope': '{:.6f}'})

**EXAM RED FLAG —** A **negative trend slope** means the paid/incurred ratio is declining across origin years — case reserves are becoming more adequate over time. Projecting case development ratios from a triangle with increasing case adequacy will systematically overstate future paid claims. Consider Berquist-Sherman reported adjustment before applying Technique #1.

**EXAM RED FLAG —** If `paid_to_incurred` is near 1.0 for all origin years and the slope is near zero, case adequacy is stable and both Technique #1 and Technique #2 are appropriate.

See `exam5_diagnostics.ipynb` Section 5 for full case adequacy diagnostic walkthrough.

## Technique #1: Develop Case Outstanding and Project Incremental Paid

This section uses two implementations:
- A Friedland-style explicit ratio implementation (`run_case_outstanding_friedland`).
- A chainladder-native implementation using `chainladder.CaseOutstanding` (`run_case_outstanding_chainladder`).

Both use case and paid triangles, but the chainladder-native path keeps more of the projection workflow inside `chainladder` objects.


In [ ]:
friedland_result, friedland_summary, friedland_artifacts = run_case_outstanding_friedland(
    case_os_triangle,
    paid_incremental_triangle,
)

co_chainladder_result, co_chainladder_summary, co_chainladder_artifacts = run_case_outstanding_chainladder(
    paid_cumulative_triangle,
    reported_cumulative_triangle,
)

ratio_table = friedland_artifacts['ratio_table'].copy()
ratio_table.head(12), friedland_summary.head(8), co_chainladder_summary.head(8)


## Technique #1 Comparison: Explicit Ratios vs Chainladder CaseOutstanding

The comparison below highlights total-level differences across implementations.
Differences are expected because the two pipelines do not estimate exactly the same internal quantity in the same way.


In [ ]:
tech1_totals = pd.DataFrame(
    [
        {
            'Method': friedland_result.method_name,
            'LatestTotal': friedland_result.latest_reported_total,
            'UltimateTotal': friedland_result.ultimate_total,
            'IBNRTotal': friedland_result.ibnr_total,
        },
        {
            'Method': co_chainladder_result.method_name,
            'LatestTotal': co_chainladder_result.latest_reported_total,
            'UltimateTotal': co_chainladder_result.ultimate_total,
            'IBNRTotal': co_chainladder_result.ibnr_total,
        },
    ]
)

tech1_chainladder_patterns = pd.DataFrame({
    'Combined_LDF': pd.Series(co_chainladder_artifacts['selected_ldfs_by_age']['combined'].get('(All)', {})),
    'Paid_to_PriorCase_LDF': pd.Series(co_chainladder_artifacts['selected_ldfs_by_age']['paid_to_prior_case'].get('(All)', {})),
    'Case_to_PriorCase_LDF': pd.Series(co_chainladder_artifacts['selected_ldfs_by_age']['case_to_prior_case'].get('(All)', {})),
})

tech1_totals, tech1_chainladder_patterns


## Technique #2: Case Outstanding with Paid/Reported CDF Combination

This method uses industry-style paid and reported CDFs to build a case outstanding development factor by maturity.

Implementation steps:
1. Fit development to paid cumulative and reported cumulative triangles.
2. Get selected paid and reported CDFs by maturity.
3. Compute Case OS Dev Factor from the CDF relationship.
4. Apply factor to latest case outstanding to project unpaid and ultimate paid.


In [ ]:
paid_dev = cl.Development(average='volume', n_periods=-1).fit_transform(paid_cumulative_triangle)
reported_dev = cl.Development(average='volume', n_periods=-1).fit_transform(reported_cumulative_triangle)

paid_cdf_by_age = cl.Chainladder().fit(paid_dev).cdf_.to_frame().iloc[0]
reported_cdf_by_age = cl.Chainladder().fit(reported_dev).cdf_.to_frame().iloc[0]

case_long = case_os_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
case_col = [c for c in case_long.columns if c not in ['Total', 'origin', 'development', 'valuation']][0]
case_matrix = case_long.pivot(index='origin', columns='development', values=case_col).sort_index().sort_index(axis=1)
latest_age_by_ay = case_matrix.notna().iloc[:, ::-1].idxmax(axis=1).astype(int)
latest_age_by_ay.index = latest_age_by_ay.index.map(lambda x: int(getattr(x, 'year', int(str(x)[:4]))))

tech2_rows = []
for idx in latest_case.index:
    ay = int(idx.year)
    age = int(latest_age_by_ay.loc[ay])
    paid_cdf = float(paid_cdf_by_age.get(f'{age}-Ult', 1.0))
    reported_cdf = float(reported_cdf_by_age.get(f'{age}-Ult', 1.0))
    denom = paid_cdf - reported_cdf
    case_os_factor = 1.0 + ((reported_cdf - 1.0) * paid_cdf / denom) if abs(denom) > 1e-12 else 1.0

    latest_case_val = float(latest_case.loc[idx])
    latest_paid_val = float(latest_paid.loc[idx])
    latest_incurred_val = float(latest_reported.loc[idx])
    projected_unpaid = latest_case_val * case_os_factor
    projected_ultimate_paid = latest_paid_val + projected_unpaid
    additional_ibnr_above_case = projected_unpaid - latest_case_val

    tech2_rows.append(
        {
            'AccidentYear': ay,
            'LatestAge': age,
            'PaidCDF': paid_cdf,
            'ReportedCDF': reported_cdf,
            'CaseOSDevFactor': case_os_factor,
            'LatestCaseOutstanding': latest_case_val,
            'LatestPaidCumulative': latest_paid_val,
            'LatestReportedCumulative': latest_incurred_val,
            'ProjectedUnpaid': projected_unpaid,
            'ProjectedUltimatePaid': projected_ultimate_paid,
            'AdditionalIBNRAboveCase': additional_ibnr_above_case,
        }
    )

tech2_projection = pd.DataFrame(tech2_rows).set_index('AccidentYear').sort_index()
tech2_projection.loc['Total'] = tech2_projection.sum()
tech2_projection


## Technique #2 Diagnostic Tables

The tables below show the paid/reported CDF curves used in the combination formula, and the resulting case outstanding development factor by age.


In [ ]:
cdf_compare = pd.DataFrame(
    {
        'PaidCDF': paid_cdf_by_age,
        'ReportedCDF': reported_cdf_by_age,
    }
)
cdf_compare['CaseOSDevFactor'] = 1.0 + ((cdf_compare['ReportedCDF'] - 1.0) * cdf_compare['PaidCDF']) / (cdf_compare['PaidCDF'] - cdf_compare['ReportedCDF'])
cdf_compare


## Final Comparison and Assumptions

Key assumptions for case outstanding methods:
1. Case adequacy and claims processing remain reasonably stable.
2. Paid/reporting development patterns are representative for future emergence.
3. Mix, limits, and retention structure are sufficiently stable (or segmented).

Technique tradeoff summary:
- Technique #1 is process-intuitive and can be very useful when case and paid interaction is stable.
- Technique #2 is more benchmark-driven via paid/reported CDFs and can be robust when explicit case ratio calibration is noisy.


## Red-Flag Notes for Exam 5

**EXAM RED FLAG —** Technique #2 formula breaks down when Paid CDF = Reported CDF (denominator = 0). This occurs only if paid and reported develop at exactly the same rate — unusual. On the exam, if you compute Paid CDF = Reported CDF at any age, verify your CDF calculations before concluding the technique is undefined.

**EXAM RED FLAG —** Technique #1 is unstable when the case OS at the latest age is near zero (nearly fully paid). The remaining-in-case ratio r_case(d) and paid-on-case ratio r_paid(d) are both computed relative to the prior age's case OS — if that is very small, the ratios are noisy. Always check whether the latest case OS balance is material before applying Technique #1.

**EXAM RED FLAG —** If you apply Technique #1 without Berquist-Sherman on a triangle where case adequacy has been increasing, your selected r_case(d) ratios will blend periods of different adequacy levels. The most recent ratios reflect the strongest adequacy; older ratios reflect weaker adequacy. The vol-wtd average will produce a blend that does not reflect the current adequacy level.

### Method Leverage

The most recent accident year in Technique #1 carries the highest projection leverage: if it is at age 12 months (one diagonal), ALL future paid development is projected from the current case OS balance — a single case reserve estimate drives the entire reserve for that year. Verify that the 12-month case OS is credible before projecting.

In [ ]:
final_comparison = pd.DataFrame(
    [
        {
            'Method': friedland_result.method_name,
            'ProjectedUltimate': friedland_result.ultimate_total,
            'ProjectedIBNR': friedland_result.ibnr_total,
            'ReferenceLatest': friedland_result.latest_reported_total,
        },
        {
            'Method': co_chainladder_result.method_name,
            'ProjectedUltimate': co_chainladder_result.ultimate_total,
            'ProjectedIBNR': co_chainladder_result.ibnr_total,
            'ReferenceLatest': co_chainladder_result.latest_reported_total,
        },
        {
            'Method': 'Case Outstanding Technique #2 (CDF Combination)',
            'ProjectedUltimate': float(tech2_projection.loc['Total', 'ProjectedUltimatePaid']),
            'ProjectedIBNR': float(tech2_projection.loc['Total', 'ProjectedUnpaid']),
            'ReferenceLatest': float(tech2_projection.loc['Total', 'LatestPaidCumulative']),
        },
    ]
)

impact_table = pd.DataFrame(
    [
        [
            'Speedup in settlement rate',
            'r_paid(d) ratios capture historical speed — if speed increased recently, ratios are understated; future paid will be underestimated',
            'CDF_paid increases (faster settlement) — Case OS Dev Factor adjusts, but if change is mid-triangle, CDFs blend periods',
        ],
        [
            'Increase in case outstanding adequacy',
            'r_case(d) and r_paid(d) both shift — adequacy increase means lower r_paid and higher r_case; vol-wtd average blends inconsistent periods',
            'CDF_rep increases (reported develops faster) — formula adjusts, but CDFs blend periods if adequacy changed mid-triangle',
        ],
        [
            'Average accident date shifts forward',
            'More immature claims at each diagonal — higher remaining case ratios at each age (less mature)',
            'Higher CDFs for same development age',
        ],
        [
            'Change in product mix',
            'Ratios need to be segmented if new mix has materially different case and paid patterns',
            'CDFs need to be re-selected by segment if mix changes materially',
        ],
        [
            'Berquist-Sherman adjustment already applied',
            'Ratios derived from adjusted triangle reflect consistent case adequacy — preferred approach',
            'CDFs derived from adjusted triangles — preferred when distortion existed',
        ],
        [
            'Case OS near zero at latest age',
            'Technique #1 is unstable — ratios r_case and r_paid become noisy near zero; use Technique #2',
            'Technique #2 less sensitive to individual case balance levels — preferred in this scenario',
        ],
    ],
    columns=['Change in Environment', 'Impact on Technique #1', 'Impact on Technique #2'],
)

final_comparison, impact_table

# Exam 5 Practice Problems — Case Outstanding Methods

Work through each problem by hand before running the solution cell.
This notebook covers **two distinct methods** that both use case outstanding as their anchor:

| Method | Core Formula | Key Input |
|---|---|---|
| **Development** | Ultimate = Case OS × Factor | Factor = Ult / Case OS (age-specific) |
| **Additive** | IBNR = Case OS × Ratio | Ratio = IBNR / Case OS (uniform) |

Both methods project IBNR using *current* case outstanding — so case strengthening or weakening directly affects both methods' projections.

## Practice Problem 1: Case Outstanding Development Method

You are given the following data as of December 31, 2024:

| AY | Age (mo) | Case OS | Reported to Date |
|---|---|---|---|
| 2022 | 48 | 6,000 | 40,000 |
| 2023 | 36 | 10,000 | 34,000 |
| 2024 | 24 | 16,000 | 28,000 |

Selected age-to-ultimate **Ult/CaseOS** factors (from historical development):

| Age | Factor |
|---|---|
| 24 months | 3.30 |
| 36 months | 5.00 |
| 48 months | 7.50 |

**(a)** Calculate the estimated ultimate losses for each AY.

**(b)** Calculate IBNR for each AY.

**(c)** Which AY has the highest IBNR as a percent of reported losses? Explain why this is expected.

**(d)** Case outstanding for AY 2024 has been strengthened over the past year. What is the directional impact on the Development Method's IBNR estimate? Is this a strength or weakness of the method?

In [ ]:
import pandas as pd

# ── Data ──────────────────────────────────────────────────────────────
data = {
    'AY':       [2022, 2023, 2024],
    'Age':      [48,   36,   24],
    'CaseOS':   [6_000, 10_000, 16_000],
    'Reported': [40_000, 34_000, 28_000],
    'Factor':   [7.50,   5.00,   3.30],   # Ult / CaseOS at that age
}
df = pd.DataFrame(data)

# ── (a) Ultimate = Case OS × Factor ───────────────────────────────────
df['Ultimate'] = df['CaseOS'] * df['Factor']

# ── (b) IBNR = Ultimate − Reported ────────────────────────────────────
df['IBNR'] = df['Ultimate'] - df['Reported']

# ── (c) IBNR % of Reported ────────────────────────────────────────────
df['IBNR_pct_Reported'] = df['IBNR'] / df['Reported']

print('=== Case Outstanding Development Method ===')
print(df[['AY','Age','CaseOS','Reported','Factor','Ultimate','IBNR','IBNR_pct_Reported']].to_string(index=False))

print()
print('--- Key Checks ---')
print('IBNR = Ultimate - Reported  (NOT just Ultimate)')
print('Totals:', df['IBNR'].sum(), 'total IBNR')

print()
print('--- Part (c): Highest IBNR % of Reported ---')
max_row = df.loc[df['IBNR_pct_Reported'].idxmax()]
print(f'AY {int(max_row.AY)} has highest IBNR/Reported = {max_row.IBNR_pct_Reported:.1%}')
print('Expected: youngest AY (24 mo) is least developed; most losses are still unreported.')

print()
print('--- Part (d): Case Strengthening Impact ---')
print('Strengthening Case OS INCREASES the Development Method Ultimate.')
print('  Ultimate = CaseOS x Factor; higher CaseOS -> higher projected Ultimate.')
print('  This is a weakness: the method cannot distinguish a genuinely worse AY')
print('  from one where adjusters simply raised case reserves.')
print('  It can lead to double-counting: case OS rises AND factor stays high.')


## Practice Problem 2: Case Outstanding Additive Method

Using the same data as Problem 1 (Case OS and Reported below), apply the **Additive Method** with a selected IBNR/CaseOS ratio of **0.85**.

| AY | Age (mo) | Case OS | Reported to Date |
|---|---|---|---|
| 2022 | 48 | 6,000 | 40,000 |
| 2023 | 36 | 10,000 | 34,000 |
| 2024 | 24 | 16,000 | 28,000 |

**(a)** Calculate IBNR for each AY using the Additive Method.

**(b)** Calculate the estimated ultimate losses.

**(c)** Compare the ultimates from the Development Method (Problem 1) and the Additive Method. Which gives a higher IBNR for AY 2022? Explain why.

**(d)** A critic argues the Additive Method unfairly penalizes AYs with large case reserves. Is this valid? When would you prefer the Additive Method over the Development Method?

In [ ]:
import pandas as pd

# ── Data ──────────────────────────────────────────────────────────────
data = {
    'AY':       [2022, 2023, 2024],
    'Age':      [48,   36,   24],
    'CaseOS':   [6_000, 10_000, 16_000],
    'Reported': [40_000, 34_000, 28_000],
}
df = pd.DataFrame(data)

ratio = 0.85   # selected IBNR / CaseOS ratio

# ── (a) IBNR = CaseOS × ratio ─────────────────────────────────────────
df['IBNR'] = df['CaseOS'] * ratio

# ── (b) Ultimate = Reported + IBNR ────────────────────────────────────
df['Ultimate'] = df['Reported'] + df['IBNR']

# Development Method ultimates (for comparison)
dev_ultimates = [45_000, 50_000, 52_800]
dev_ibnr      = [5_000,  16_000, 24_800]
df['Dev_Ultimate'] = dev_ultimates
df['Dev_IBNR']     = dev_ibnr

print('=== Case Outstanding Additive Method ===')
print(f'Selected IBNR/CaseOS ratio: {ratio}')
print(df[['AY','Age','CaseOS','Reported','IBNR','Ultimate']].to_string(index=False))

print()
print('=== Comparison: Development vs Additive ===')
cmp = df[['AY','Dev_Ultimate','Ultimate','Dev_IBNR','IBNR']].copy()
cmp.columns = ['AY','Dev_Ultimate','Add_Ultimate','Dev_IBNR','Add_IBNR']
print(cmp.to_string(index=False))

print()
print('--- Part (c): AY 2022 comparison ---')
print(f'  Development IBNR: {dev_ibnr[0]:,.0f}  |  Additive IBNR: {df.loc[0,"IBNR"]:,.0f}')
print('  Development is higher because it multiplies CaseOS by a LARGE factor (7.5 at 48 mo).')
print('  Additive uses a FLAT ratio (0.85); the same ratio applied to a small CaseOS at 48 mo')
print('  gives much less IBNR than a high age-specific factor.')

print()
print('--- Part (d): Additive vs Development ---')
print('Critic is partially right: Additive IBNR is proportional to current CaseOS,')
print('  so case-strengthened AYs get MORE projected IBNR (same direction as Development).')
print('Additive is PREFERRED when:')
print('  - Age-specific factors are unstable or based on thin data')
print('  - You believe a single ratio captures the IBNR relationship across ages')
print('  - You want to limit the amplification from large age-specific factors')
print('Development is PREFERRED when:')
print('  - Age-specific factors are stable and credible')
print('  - IBNR/CaseOS clearly varies by age (immature vs mature AYs differ structurally)')


## Conceptual Reference: When Each Method Works and Fails

### Key Relationships

| | Development Method | Additive Method |
|---|---|---|
| Formula | Ultimate = CaseOS × Factor | IBNR = CaseOS × Ratio |
| Factor varies by | Age (decreasing toward tail) | Same for all AYs |
| Case strengthening → | Higher Ultimate (amplified) | Higher IBNR (proportional) |
| Case weakening → | Lower Ultimate (deflated) | Lower IBNR |
| Best when | Stable historical development patterns | Thin age-specific data, or ratio stable |
| Fails when | Case reserving philosophy changes | IBNR/CaseOS varies significantly by age |

### Case Strengthening / Weakening — Quick Check

**Strengthening** (case reserves increased without new information):
- Historical `Ult/CaseOS` factors were calibrated when reserves were lower
- Applying those same high factors to now-inflated CaseOS → **overstated Ultimate**
- Additive: same issue — current CaseOS is inflated → **overstated IBNR**
- Neither method self-corrects automatically for a change in reserving philosophy

**Weakening** (case reserves released too early):
- Historical factors calibrated on fuller reserves
- Current CaseOS is too low → both methods project **understated Ultimate**

### Exam Memory Aid

> Both methods anchor on **current** case outstanding, so any systematic bias in case reserving flows directly into the projection — this is the primary weakness of both methods.

### When to Prefer Case OS Methods Over Chain Ladder / BF

- Triangle data is sparse or unavailable (case data is more readily available)
- You trust the current case reserve levels as a signal of ultimate
- Claim severity rather than claim count is the main driver of reserve variability
- Short-tail lines where most information is in open claim files


## Exam-Style Written Answer Examples

### Example A — Compare Development and Additive results for the same AY

**Prompt:** The Development Method projects AY 2024 ultimate at $52,800 while the Additive Method projects $41,600. Explain the source of this difference and which method you would rely on more for this AY.

**Weak answer (partial credit):** "The Development Method uses age-specific factors and the Additive uses a flat ratio, so the results differ."

**Strong answer (full credit):** "The Development Method applies a factor of 3.3 — the ratio of ultimate to case outstanding at 24 months — to AY 2024's current case OS. This factor is high because at 24 months, the case OS represents only a small fraction of ultimate. The Additive Method applies a lower, age-independent ratio and adds that IBNR to reported. The Development Method will produce a higher estimate whenever the age-specific factor exceeds the uniform ratio. For a 24-month AY I would scrutinize whether the historical factor is stable; if yes, prefer Development. If data is sparse, the Additive Method's uniform ratio may be more stable."

---

### Example B — Case strengthening impact

**Prompt:** Claim adjusters increased case reserves across all open claims in AY 2023 by 20% at year-end. You are using the Development Method. What is the directional effect on the projected ultimate and on IBNR? Is this a concern?

**Weak answer:** "Higher case reserves mean higher ultimate."

**Strong answer:** "The Development Method computes Ultimate = CaseOS × Factor. A 20% increase in CaseOS for AY 2023 produces a 20% increase in projected ultimate (since the factor is held constant). However, Reported already includes the strengthened case reserves, so IBNR = Ultimate − Reported increases by the factor × 20% × original CaseOS minus the 20% increase in case (which is already in Reported). The concern is that the historical factors were calibrated when case reserves followed the prior philosophy. Applying them to inflated current case OS effectively double-counts the strengthening: once in the higher factor numerator (if it was also calculated using strengthened data) and once in current CaseOS. The actuary should check whether factors were derived from data with the same reserving philosophy."

---

### Example C — Recommend a method for a book with unstable case reserving

**Prompt:** Claim management recently changed, and average case reserves have been volatile over the past three years. Should you use the Case Outstanding Development Method? If not, what would you use instead?

**Strong answer:** "The Case Outstanding Development Method relies on stable historical ratios of ultimate to case outstanding. If case reserving philosophy has been volatile, the historical factors are not representative of current reserve levels, and applying them to current case OS will produce unreliable ultimates. I would instead consider the Bornhuetter-Ferguson or Expected Claims method (less sensitive to current data), or the Chain Ladder method applied to paid losses (bypasses case reserve volatility). If I did use the Case Outstanding method, I would first re-estimate the factors using only periods with a consistent reserving philosophy."

---

### Example D — When are the two case outstanding methods equivalent?

**Strong answer:** "The methods give the same IBNR when the historical IBNR/CaseOS ratio is constant across all ages — i.e., when the amount of IBNR relative to case OS does not change as claims mature. In practice this rarely holds; younger AYs typically have more IBNR per dollar of case outstanding than mature AYs, so the Development Method (with age-specific factors) will project more IBNR for immature AYs than the Additive Method with a single blended ratio."


## Common Pitfalls / Exam Trap Awareness

### Mechanical Traps

| Trap | How to Avoid |
|---|---|
| Using the same factor for all ages | Factors are **age-specific** in the Development Method |
| Confusing the two formulas | Dev: `Ultimate = CaseOS × Factor`; Add: `IBNR = CaseOS × Ratio` |
| Forgetting IBNR = Ultimate − Reported (Dev) | Always subtract Reported after computing Ultimate |
| Applying Additive ratio to Reported instead of CaseOS | Ratio applies to **Case OS**, not to Reported |
| Treating a large case OS as inherently bad | Case OS is just an input; size alone doesn't signal over/under |
| Selecting factor from only one historical year | Factors need credibility; use volume-weighted averages |

### Judgment / Written Answer Traps

| Trap | What Examiners Want |
|---|---|
| Saying 'case strengthening increases IBNR' without direction | State: strengthening → higher CaseOS → higher Ultimate (Dev) AND higher IBNR (both methods) |
| Choosing one method without comparing | Name the specific advantage of the chosen method for this situation |
| Ignoring that Reported includes case reserves | Reported = Paid + Case OS; changing case OS changes Reported too |
| Saying CL is always better than Case OS methods | Case OS methods can be superior when triangle data is sparse |
| Mixing Development and Additive formulas in the same problem | Pick one method and be consistent throughout |

### Self-Check Before Finalizing

- [ ] Did I use the correct formula for the method specified?
- [ ] Are my factors age-specific (Development) or uniform (Additive)?
- [ ] IBNR = Ultimate − Reported (Development) **or** IBNR = CaseOS × Ratio (Additive)?
- [ ] Did I check whether case reserving philosophy is consistent across years?
- [ ] For written answers: did I state the direction of the effect (not just 'it changes')?
- [ ] Did I address whether the assumption underlying the chosen method is met?
- [ ] Are there any negative IBNR values? (Flag as a potential problem.)
- [ ] Have I connected the method choice to the characteristics of the data/line?
